# Lab: create and register agents on AWS Agent Registry and Gemini Enterprise Agent Platform

This notebook walks through the two managed agent registries that ShadowScan operators most often
need to reconcile against: **AWS Agent Registry** (part of Amazon Bedrock AgentCore) and the
**Agent Registry of Gemini Enterprise Agent Platform** (Google Cloud, formerly Vertex AI). On each
platform you will

1. create a small agent (optional, because it costs money and takes a few minutes),
2. describe it with an A2A agent card,
3. register it in the platform's registry and move it through the approval / discovery lifecycle,
4. discover it the way a consumer (another agent, an IDE, a gateway) would, and
5. look at what ShadowScan can and cannot observe, then bind the agent in a sanctioned inventory.

Nothing here is specific to ShadowScan until Part C. Parts A and B are plain platform usage that you
can reuse for your own agents.

| | AWS Agent Registry | Gemini Enterprise Agent Platform |
|---|---|---|
| Service | `agent-registry` (control plane `agent-registry-control`, data plane `agent-registry`) | `agentregistry.googleapis.com` (Agent Registry API v1) |
| Catalog object | **Registry** (per account and Region) holding **records** | Per **project and location**; you write a **Service**, the registry derives a read-only **Agent**, **McpServer** or **Endpoint** |
| Record kinds | `AGENT` (A2A card), `MCP`, `SKILL`, `CUSTOM`, `GATEWAY` | agent (`A2A_AGENT_CARD` or `NO_SPEC` + interfaces), MCP server (`TOOL_SPEC`), endpoint |
| Lifecycle | `DRAFT` → `PENDING_APPROVAL` → `APPROVED` (`REJECTED`, `DEPRECATED`); only approved records are discoverable | Created, then listed and searchable at once; governance through IAM, Agent Gateway and policies |
| Automatic population | Auto-detection of AgentCore Runtimes and Gateways across an AWS Organization | Automatic registration of agents deployed to Agent Runtime (formerly Agent Engine) |
| Consumer access | `SearchDiscoverableRegistryRecords`, `ListDiscoverableRegistryRecords`, registry MCP endpoint (IAM or custom JWT authorizer) | `agents.list`, `agents.search`, Agent Registry MCP server, ADK integration |
| Runtime used in this lab | Amazon Bedrock AgentCore Runtime (A2A protocol, port 9000) | Agent Runtime (`reasoningEngines`), ADK agent |
| End-user surface | Registry MCP endpoint in IDEs and agents | Gemini Enterprise app (register through the Discovery Engine API) |

**Read before you run.** The notebook creates billable cloud resources in whatever account and
project your credentials point at. Use a sandbox account and project, read the cleanup section
(Part D) first, and never paste credentials into a cell. All settings come from environment
variables so that the notebook can be committed without edits.

## Prerequisites

* Python 3.11 or newer with a Jupyter kernel.
* Packages: `boto3>=1.43.68` (first release with the `agent-registry-control` client),
  `google-cloud-agentregistry>=0.1.2`, `google-auth`, `requests`, `PyYAML`.
* Optional, only for the "create the agent" steps: `bedrock-agentcore-starter-toolkit` (Part A.1) and
  `google-cloud-aiplatform[agent_engines,adk]` plus `google-adk` (Part B.1).
* Optional for Part C: ShadowScan installed from a reviewed checkout (`pip install -e '.[aws,gcp]'`).

```bash
pip install "boto3>=1.43.68" "google-cloud-agentregistry>=0.1.2" google-auth requests PyYAML
pip install bedrock-agentcore-starter-toolkit                      # Part A.1
pip install "google-cloud-aiplatform[agent_engines,adk]" google-adk  # Part B.1
```

### Permissions

**AWS.** A principal in a sandbox account with, at minimum, the Agent Registry actions used below.
Scope the resource to `arn:aws:agent-registry:<region>:<account>:registry/*`.

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "RegistryLab",
      "Effect": "Allow",
      "Action": [
        "agent-registry:CreateRegistry", "agent-registry:GetRegistry", "agent-registry:ListRegistries",
        "agent-registry:UpdateRegistry", "agent-registry:DeleteRegistry",
        "agent-registry:CreateRegistryRecord", "agent-registry:GetRegistryRecord",
        "agent-registry:ListRegistryRecords", "agent-registry:UpdateRegistryRecord",
        "agent-registry:UpdateRegistryRecordStatus", "agent-registry:DeleteRegistryRecord",
        "agent-registry:SubmitRegistryRecordForApproval",
        "agent-registry:TagResource", "agent-registry:ListTagsForResource",
        "agent-registry:SearchDiscoverableRegistryRecords",
        "agent-registry:ListDiscoverableRegistryRecords",
        "agent-registry:BatchGetDiscoverableRegistryRecord"
      ],
      "Resource": "arn:aws:agent-registry:*:123456789012:registry/*"
    }
  ]
}
```

The optional AgentCore deployment (Part A.1) additionally needs what the starter toolkit documents:
`bedrock-agentcore-control` runtime operations, IAM role creation for the execution role, S3 for the
code package and CodeBuild. Grant that only in the sandbox.

**Google Cloud.** On a sandbox project: *Agent Registry API Admin* (`roles/agentregistry.admin`) for
Part B, *Service Usage Admin* to enable APIs, and for the optional deployment *Vertex AI User*
(`roles/aiplatform.user`) plus write access to the staging bucket. Registering into a Gemini
Enterprise app (Part B.4) needs Discovery Engine admin rights on that app.

```bash
gcloud services enable agentregistry.googleapis.com aiplatform.googleapis.com discoveryengine.googleapis.com
gcloud auth application-default login
```

### Run modes (environment variables)

| Variable | Default | Effect |
|---|---|---|
| `LAB_DRY_RUN` | `0` | `1` validates every request against the SDK models and prints it, without calling any cloud API. Useful for a first read-through and for CI. |
| `AWS_REGION` | `us-east-1` | Region for Part A. Agent Registry launched in `us-east-1`, `us-west-2`, `eu-west-1`, `ap-northeast-1` and `ap-southeast-2`. |
| `AWS_DEPLOY_AGENT` | `0` | `1` deploys the lab agent to AgentCore Runtime (Part A.1). |
| `AWS_RUNTIME_ARN` | empty | Reuse an existing AgentCore runtime instead of deploying one. |
| `LAB_AUTO_APPROVE` | `1` | Create the AWS registry with `APPROVE_ALL` auto-approval. `0` keeps manual curation. |
| `LAB_ACT_AS_CURATOR` | `1` | Approve a `PENDING_APPROVAL` record yourself in Part A.4. |
| `GOOGLE_CLOUD_PROJECT`, `GOOGLE_CLOUD_LOCATION` | ADC project, `us-central1` | Project and location for Part B. |
| `GCP_DEPLOY_AGENT`, `GCP_STAGING_BUCKET` | `0`, empty | `1` deploys an ADK agent to Agent Runtime; needs a `gs://` staging bucket. |
| `GCP_ENGINE_NAME` | empty | Reuse an existing `projects/.../reasoningEngines/...` resource. |
| `GE_APP_ID`, `GE_LOCATION`, `GE_PROJECT_NUMBER` | empty, `global`, project id | Gemini Enterprise app for Part B.4; empty skips it. |
| `RUN_SHADOWSCAN` | `0` | `1` runs ShadowScan's cloud connectors in Part C with a separate, read-only identity. |
| `LAB_CLEANUP` | `0` | `1` deletes everything the notebook created (Part D). |
| `LAB_PREFIX`, `LAB_RUN_ID` | `nexus-lab`, random | Naming of everything created. |

In [ ]:
import importlib
import sys

REQUIRED = {
    "boto3": "boto3>=1.43.68",
    "botocore": "botocore (installed with boto3)",
    "google.cloud.agentregistry_v1": "google-cloud-agentregistry>=0.1.2",
    "google.auth": "google-auth",
    "requests": "requests",
    "yaml": "PyYAML",
}
OPTIONAL = {
    "bedrock_agentcore_starter_toolkit": "bedrock-agentcore-starter-toolkit (Part A.1)",
    "agentplatform": "google-cloud-aiplatform[agent_engines,adk] (Part B.1)",
    "google.adk": "google-adk (Part B.1)",
    "shadowscan": "project-nexus-shadowscan (Part C)",
}

print(f"Python {sys.version.split()[0]}")
missing = []
for group, modules in (("required", REQUIRED), ("optional", OPTIONAL)):
    for module, package in modules.items():
        try:
            importlib.import_module(module)
            state = "ok"
        except ImportError:
            state = "missing"
            if group == "required":
                missing.append(package)
        print(f"{state:8s} {group:9s} {module:36s} {package}")
if missing:
    print("\nInstall the missing required packages and restart the kernel:")
    print("  pip install " + " ".join(f'"{package}"' for package in missing))

## Configuration

Everything is read from the environment. Review the printed table before continuing: it is the only
place where account, project and naming choices are made.

In [ ]:
import os
import re
import uuid
from pathlib import Path


def env_flag(name: str, default: bool = False) -> bool:
    """Read an on/off switch from the environment (1, true, yes, on)."""
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "on"}


DRY_RUN = env_flag("LAB_DRY_RUN")
RUN_ID = re.sub(r"[^a-z0-9]", "", os.environ.get("LAB_RUN_ID", uuid.uuid4().hex[:6]).lower()) or "lab"
PREFIX = re.sub(r"[^a-z0-9-]", "-", os.environ.get("LAB_PREFIX", "nexus-lab").lower()).strip("-")
PREFIX = PREFIX or "nexus-lab"
OUTPUT_DIR = Path(os.environ.get("LAB_OUTPUT_DIR", "lab-output")).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Part A: AWS Agent Registry
AWS_REGION = os.environ.get("AWS_REGION") or os.environ.get("AWS_DEFAULT_REGION") or "us-east-1"
AWS_DEPLOY_AGENT = env_flag("AWS_DEPLOY_AGENT")
AWS_AUTO_APPROVE = env_flag("LAB_AUTO_APPROVE", default=True)
AWS_ACT_AS_CURATOR = env_flag("LAB_ACT_AS_CURATOR", default=True)
AWS_A2A_SCHEMA_VERSION = os.environ.get("AWS_A2A_SCHEMA_VERSION", "0.3")

# Part B: Gemini Enterprise Agent Platform
GCP_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT") or ""
GCP_LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
GCP_DEPLOY_AGENT = env_flag("GCP_DEPLOY_AGENT")
GCP_STAGING_BUCKET = os.environ.get("GCP_STAGING_BUCKET", "")
GCP_MODEL = os.environ.get("GCP_MODEL", "gemini-2.5-flash")
GE_APP_ID = os.environ.get("GE_APP_ID", "")
GE_LOCATION = os.environ.get("GE_LOCATION", "global")
GE_PROJECT_NUMBER = os.environ.get("GE_PROJECT_NUMBER", "")

# Parts C and D
RUN_SHADOWSCAN = env_flag("RUN_SHADOWSCAN")
LAB_CLEANUP = env_flag("LAB_CLEANUP")

AGENT_NAME = f"{PREFIX}-agent-{RUN_ID}"
REGISTRY_NAME = f"{PREFIX}-registry-{RUN_ID}"

settings = {
    "DRY_RUN": DRY_RUN,
    "RUN_ID": RUN_ID,
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "AWS_REGION": AWS_REGION,
    "AWS_DEPLOY_AGENT": AWS_DEPLOY_AGENT,
    "AWS_AUTO_APPROVE": AWS_AUTO_APPROVE,
    "GCP_PROJECT": GCP_PROJECT or "(from Application Default Credentials)",
    "GCP_LOCATION": GCP_LOCATION,
    "GCP_DEPLOY_AGENT": GCP_DEPLOY_AGENT,
    "GE_APP_ID": GE_APP_ID or "(skip Part B.4)",
    "RUN_SHADOWSCAN": RUN_SHADOWSCAN,
    "LAB_CLEANUP": LAB_CLEANUP,
    "AGENT_NAME": AGENT_NAME,
    "REGISTRY_NAME": REGISTRY_NAME,
}
for key, value in settings.items():
    print(f"{key:18s} {value}")

## Shared helpers

`build_agent_card` produces an [A2A protocol](https://a2a-protocol.org/latest/specification/) 0.3
agent card. Both registries accept the card as the canonical description of an agent: AWS stores it
as the `AGENT` record descriptor, Google validates it as the `A2A_AGENT_CARD` service spec and
derives the read-only `Agent` resource (display name, skills, protocols) from it.

In [ ]:
import json
import time
from collections.abc import Callable
from typing import Any


def pretty(value: Any, limit: int = 4000) -> None:
    """Print a JSON rendering of an API response, truncated for readability."""
    text = json.dumps(value, indent=2, sort_keys=True, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n... (truncated)")


def wait_for(
    describe: Callable[[], dict[str, Any]],
    done: Callable[[dict[str, Any]], bool],
    *,
    what: str,
    timeout: float = 600,
    interval: float = 5,
) -> dict[str, Any]:
    """Poll ``describe`` until ``done`` says so or ``timeout`` seconds pass; prints status changes."""
    deadline = time.monotonic() + timeout
    last = None
    while True:
        state = describe()
        status = state.get("status") or state.get("state")
        if status != last:
            print(f"{what}: {status}")
            last = status
        if done(state):
            return state
        if time.monotonic() > deadline:
            raise TimeoutError(f"{what} did not finish within {timeout:.0f}s (last status {status})")
        time.sleep(0 if DRY_RUN else interval)


def build_agent_card(
    *, name: str, description: str, url: str, version: str, skill_tags: list[str]
) -> dict[str, Any]:
    """Return a minimal A2A 0.3 agent card for the lab agent."""
    return {
        "protocolVersion": "0.3.0",
        "name": name,
        "description": description,
        "url": url,
        "preferredTransport": "JSONRPC",
        "version": version,
        "provider": {
            "organization": "Project Nexus registry lab",
            "url": "https://github.com/aisecnomad/Project-Nexus",
        },
        "capabilities": {"streaming": False, "pushNotifications": False, "stateTransitionHistory": False},
        "defaultInputModes": ["text/plain"],
        "defaultOutputModes": ["text/plain"],
        "skills": [
            {
                "id": "lab-echo",
                "name": "Lab echo",
                "description": "Answers short questions about the registry lab. Demonstration only.",
                "tags": skill_tags,
                "examples": ["Which registry are you listed in?"],
            }
        ],
    }


print("helpers ready")

# Part A: AWS Agent Registry

AWS Agent Registry is the governed catalog of Amazon Bedrock AgentCore. It became generally
available in August 2026 under its own `agent-registry` service namespace (endpoints, IAM actions
and ARNs); the public-preview `bedrock-agentcore` namespace is being retired. The model:

* A **registry** is an account- and Region-scoped catalog. It carries a *discovery configuration*
  (who may search it: `AWS_IAM` or a `CUSTOM_JWT` authorizer), an *approval configuration*
  (`APPROVE_ALL` auto-approval or manual curation), optional custom-metadata schemas, optional KMS
  encryption and optional *auto-detection* of AgentCore Runtimes and Gateways across the Organization.
* A **record** describes one resource. `recordType` selects the descriptor: `AGENT` carries an A2A
  agent card (`descriptors.a2aAgentCard`), `MCP` a server descriptor plus tool list, `SKILL` an agent
  skills definition, `CUSTOM` your own JSON and `GATEWAY` an AgentCore Gateway. Descriptor content is
  inline or synchronized from an HTTPS URL (`source.fromUrl`, optionally with an IAM or OAuth
  credential provider).
* Records are versioned (`recordVersion`) and move through `CREATING` → `DRAFT` →
  `PENDING_APPROVAL` → `APPROVED`, with `REJECTED` and `DEPRECATED` as curator outcomes. **Only
  approved records are visible on the data plane** (search, list, batch get, MCP endpoint).

Two boto3 clients are involved: `agent-registry-control` (publishers and curators) and
`agent-registry` (consumers). Both sign with the `agent-registry` service name.

In [ ]:
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from botocore.validate import ParamValidator

BOTO_CONFIG = Config(
    region_name=AWS_REGION,
    retries={"max_attempts": 5, "mode": "standard"},
    connect_timeout=10,
    read_timeout=60,
)
sts = boto3.client("sts", config=BOTO_CONFIG)
registry_control = boto3.client("agent-registry-control", config=BOTO_CONFIG)  # publisher and curator API
registry_data = boto3.client("agent-registry", config=BOTO_CONFIG)  # consumer (discovery) API

LAUNCH_REGIONS = {"us-east-1", "us-west-2", "eu-west-1", "ap-northeast-1", "ap-southeast-2"}
if AWS_REGION not in LAUNCH_REGIONS:
    print(f"note: {AWS_REGION} was not in the launch Region list; check the AWS Regional Services List")


_DRY_RUN_STATE = {"record_status": "DRAFT"}


def _dry_run_response(method: str, params: dict[str, Any]) -> dict[str, Any]:
    """Shape-compatible stand-ins so the notebook can run end to end without credentials."""
    account = "123456789012"
    registry_arn = f"arn:aws:agent-registry:{AWS_REGION}:{account}:registry/DRYRUNREG0001"
    record_arn = f"{registry_arn}/record/DRYRUNREC001"
    if method == "submit_registry_record_for_approval":
        _DRY_RUN_STATE["record_status"] = "APPROVED" if AWS_AUTO_APPROVE else "PENDING_APPROVAL"
    elif method == "update_registry_record_status":
        _DRY_RUN_STATE["record_status"] = str(params.get("status", ""))
    responses: dict[str, dict[str, Any]] = {
        "get_caller_identity": {"Account": account, "Arn": f"arn:aws:iam::{account}:user/lab"},
        "create_registry": {"registryArn": registry_arn},
        "get_registry": {"registryId": "DRYRUNREG0001", "registryArn": registry_arn, "status": "READY"},
        "create_registry_record": {"recordArn": record_arn, "status": "CREATING"},
        "get_registry_record": {
            "recordArn": record_arn,
            "recordId": "DRYRUNREC001",
            "name": params.get("recordId", ""),
            "recordType": "AGENT",
            "recordVersion": "1.0.0",
            "status": _DRY_RUN_STATE["record_status"],
        },
        "submit_registry_record_for_approval": {
            "recordId": "DRYRUNREC001",
            "status": _DRY_RUN_STATE["record_status"],
        },
        "update_registry_record_status": {
            "recordId": "DRYRUNREC001",
            "status": _DRY_RUN_STATE["record_status"],
        },
        "list_discoverable_registry_records": {"registryRecords": []},
        "search_discoverable_registry_records": {"registryRecords": []},
        "batch_get_discoverable_registry_record": {"registryRecords": [], "errors": []},
    }
    return responses.get(method, {})


def aws_call(client: Any, method: str, **params: Any) -> dict[str, Any]:
    """Validate ``params`` against the SDK service model, then call the API (or simulate it)."""
    operation = client.meta.method_to_api_mapping[method]
    shape = client.meta.service_model.operation_model(operation).input_shape
    report = ParamValidator().validate(params, shape)
    if report.has_errors():
        raise ValueError(report.generate_report())
    if DRY_RUN:
        print(f"[dry-run] {client.meta.service_model.service_name}.{method}")
        return _dry_run_response(method, params)
    response = getattr(client, method)(**params)
    response.pop("ResponseMetadata", None)
    return response


identity = aws_call(sts, "get_caller_identity")
AWS_ACCOUNT = identity["Account"]
print(f"account {AWS_ACCOUNT}, Region {AWS_REGION}, principal {identity['Arn']}")

## A.1 Create the agent on AgentCore Runtime (optional)

The lab agent is a [Strands Agents](https://strandsagents.com/) agent exposed through the A2A
protocol. On AgentCore Runtime an A2A server listens on **port 9000 at the root path**; the runtime
injects the public invocation URL as `AGENTCORE_RUNTIME_URL`, which the server copies into the
`url` field of the agent card it serves at `/.well-known/agent-card.json`.

Set `AWS_DEPLOY_AGENT=1` to deploy it with the AgentCore starter toolkit using *direct code deploy*
(the code is zipped to S3; no container build or Docker needed). The toolkit creates an execution
role, an S3 bucket, the runtime and its `DEFAULT` endpoint; deployment takes a few minutes. Leave
the flag off to register the agent card without a live endpoint, or set `AWS_RUNTIME_ARN` to
describe a runtime you already have.

In [ ]:
AWS_AGENT_DIR = OUTPUT_DIR / "aws-agent"
AWS_AGENT_DIR.mkdir(parents=True, exist_ok=True)

AWS_AGENT_SOURCE = """# Minimal A2A agent for the Project Nexus registry lab, run on AgentCore Runtime.

import os

from strands import Agent
from strands.multiagent.a2a import A2AServer

agent = Agent(
    name=os.environ.get("LAB_AGENT_NAME", "nexus-lab-agent"),
    description="Demonstration agent for the Project Nexus registry lab. It answers short questions.",
    system_prompt="You are a demonstration agent in a security lab. Answer in one short sentence.",
    callback_handler=None,
)

# AgentCore Runtime serves A2A on port 9000 at the root path and injects the public invocation URL as
# AGENTCORE_RUNTIME_URL; the server advertises that URL in the agent card.
server = A2AServer(
    agent=agent,
    host="0.0.0.0",
    port=9000,
    http_url=os.environ.get("AGENTCORE_RUNTIME_URL"),
    serve_at_root=True,
)

if __name__ == "__main__":
    server.serve(host="0.0.0.0", port=9000)
"""

(AWS_AGENT_DIR / "agent.py").write_text(AWS_AGENT_SOURCE, encoding="utf-8")
(AWS_AGENT_DIR / "requirements.txt").write_text("strands-agents[a2a]>=1.0\n", encoding="utf-8")
print(f"wrote {AWS_AGENT_DIR / 'agent.py'} and {AWS_AGENT_DIR / 'requirements.txt'}")

In [ ]:
from urllib.parse import quote

aws_runtime_arn = os.environ.get("AWS_RUNTIME_ARN", "")
aws_runtime = None

if AWS_DEPLOY_AGENT and not DRY_RUN:
    from bedrock_agentcore_starter_toolkit import Runtime

    aws_runtime = Runtime()
    aws_runtime.configure(
        entrypoint=str(AWS_AGENT_DIR / "agent.py"),
        agent_name=AGENT_NAME.replace("-", "_"),  # runtime names allow letters, digits and underscores
        requirements_file=str(AWS_AGENT_DIR / "requirements.txt"),
        auto_create_execution_role=True,
        auto_create_s3=True,
        deployment_type="direct_code_deploy",
        runtime_type="PYTHON_3_12",
        protocol="A2A",
        memory_mode="NO_MEMORY",
        region=AWS_REGION,
    )
    launch = aws_runtime.launch(env_vars={"LAB_AGENT_NAME": AGENT_NAME})
    aws_runtime_arn = launch.agent_arn or ""
    runtime_state = wait_for(
        lambda: aws_runtime.status().agent or {},
        lambda state: state.get("status") in {"READY", "CREATE_FAILED", "UPDATE_FAILED"},
        what="AgentCore runtime",
        timeout=900,
        interval=15,
    )
    if runtime_state.get("status") != "READY":
        raise RuntimeError(f"runtime deployment failed: {runtime_state}")
elif AWS_DEPLOY_AGENT:
    print("dry run: the AgentCore deployment is skipped")

if aws_runtime_arn:
    # Invocation URL of an AgentCore runtime; for A2A runtimes it is also the agent card URL.
    aws_agent_url = (
        f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/"
        f"{quote(aws_runtime_arn, safe='')}/invocations?qualifier=DEFAULT"
    )
    print(f"runtime {aws_runtime_arn}\nagent URL {aws_agent_url}")
else:
    # No live endpoint: the registry still accepts an inline agent card. `.invalid` never resolves.
    aws_agent_url = f"https://{PREFIX}.example.invalid/agents/{AGENT_NAME}"
    print(f"no AgentCore runtime; the record will advertise the placeholder URL {aws_agent_url}")

## A.2 Create a registry

`CreateRegistry` is asynchronous: poll `GetRegistry` until the status is `READY`. The lab registry
uses the IAM authorizer for discovery, so consumers need `agent-registry:SearchDiscoverableRegistryRecords`
or `agent-registry:InvokeRegistryMcp` on the registry ARN. With `LAB_AUTO_APPROVE=1` (default) every
submitted record is approved automatically; production registries usually omit
`approvalConfiguration` so that a curator approves each record (see A.4).

In [ ]:
registry_params: dict[str, Any] = {
    "name": REGISTRY_NAME,
    "description": "Project Nexus lab registry: agents registered by the agent-registry lab notebook.",
    "discoveryConfiguration": {"authorizerType": "AWS_IAM"},
    "tags": {"project": "nexus-lab", "run": RUN_ID},
    "clientToken": str(uuid.uuid4()),
}
if AWS_AUTO_APPROVE:
    registry_params["approvalConfiguration"] = {"autoApprovalRules": ["APPROVE_ALL"]}

created = aws_call(registry_control, "create_registry", **registry_params)
registry_arn = created["registryArn"]
registry_id = registry_arn.rsplit("/", 1)[-1]

aws_registry = wait_for(
    lambda: aws_call(registry_control, "get_registry", registryId=registry_id),
    lambda state: state["status"] in {"READY", "CREATE_FAILED"},
    what="registry",
    timeout=300,
)
if aws_registry["status"] != "READY":
    raise RuntimeError(f"registry creation failed: {aws_registry.get('statusReason')}")
print(f"registry {registry_id} ready: {registry_arn}")

## A.3 Register the agent as an `AGENT` record

The record's descriptor is the A2A agent card serialized as a JSON string. `dataSchemaVersion` names
the card schema version the service validates against (`0.3` for A2A 0.3 cards). Instead of inline
`data`, a record can point at `source.fromUrl` so the registry fetches the card from the agent's
`/.well-known/agent-card.json`, with an IAM or OAuth credential provider when the endpoint needs
authentication. Record names are unique within a registry; `recordVersion` lets you publish new
versions later. Creation is asynchronous and lands in `DRAFT`.

In [ ]:
aws_agent_card = build_agent_card(
    name=AGENT_NAME,
    description="Demonstration A2A agent created by the Project Nexus registry lab (AWS).",
    url=aws_agent_url,
    version="1.0.0",
    skill_tags=["nexus-lab", "a2a", "demo"],
)
record_params: dict[str, Any] = {
    "registryId": registry_id,
    "name": AGENT_NAME,
    "displayName": "Nexus lab agent",
    "description": "Demonstration A2A agent registered by the Project Nexus agent-registry lab.",
    "recordType": "AGENT",
    "descriptors": {
        "a2aAgentCard": {
            "data": json.dumps(aws_agent_card),
            "dataSchemaVersion": AWS_A2A_SCHEMA_VERSION,
        }
    },
    "recordVersion": "1.0.0",
    "tags": {"project": "nexus-lab", "run": RUN_ID},
    "clientToken": str(uuid.uuid4()),
}

created_record = aws_call(registry_control, "create_registry_record", **record_params)
record_arn = created_record["recordArn"]
print(f"record {record_arn}: {created_record['status']}")

aws_record = wait_for(
    lambda: aws_call(registry_control, "get_registry_record", registryId=registry_id, recordId=record_arn),
    lambda state: state["status"] not in {"CREATING", "UPDATING"},
    what="record",
    timeout=300,
)
record_id = aws_record["recordId"]
if aws_record["status"] in {"CREATE_FAILED", "UPDATE_FAILED"}:
    raise RuntimeError(f"record creation failed: {aws_record.get('statusReason')}")
pretty({key: value for key, value in aws_record.items() if key != "descriptors"})

## A.4 Approval workflow

`SubmitRegistryRecordForApproval` moves a `DRAFT` record into the registry's approval workflow. With
`APPROVE_ALL` it becomes `APPROVED` immediately; otherwise it waits in `PENDING_APPROVAL` for a
curator, who calls `UpdateRegistryRecordStatus` with `APPROVED` or `REJECTED` and a reason.
Submissions also raise Amazon EventBridge events, which is how organizations hook ticketing or
automated review into the workflow. Deprecating an approved record (`DEPRECATED`) removes it from
discovery without deleting its history.

In [ ]:
if aws_record["status"] == "DRAFT":
    submission = aws_call(
        registry_control,
        "submit_registry_record_for_approval",
        registryId=registry_id,
        recordId=record_arn,
    )
    print(f"after submission: {submission['status']}")
    if submission["status"] == "PENDING_APPROVAL" and AWS_ACT_AS_CURATOR:
        decision = aws_call(
            registry_control,
            "update_registry_record_status",
            registryId=registry_id,
            recordId=record_arn,
            status="APPROVED",
            statusReason="Reviewed in the Project Nexus registry lab",
        )
        print(f"curator decision: {decision['status']}")
    elif submission["status"] == "PENDING_APPROVAL":
        print("the record waits for a curator (LAB_ACT_AS_CURATOR=0)")

aws_record = aws_call(registry_control, "get_registry_record", registryId=registry_id, recordId=record_arn)
print(f"final status: {aws_record['status']}")

## A.5 Discover the agent as a consumer

Consumers use the `agent-registry` data plane, which only returns approved records:
`ListDiscoverableRegistryRecords` (paginated, filterable by `recordType` and `descriptorType`),
`SearchDiscoverableRegistryRecords` (natural-language search over one registry, ranked by relevance)
and `BatchGetDiscoverableRegistryRecord` (full descriptors by ID). The same registry is also reachable
as an MCP server: the console shows the endpoint URL, callers need `agent-registry:InvokeRegistryMcp`
plus the search permission, and tools such as `search_discoverable_registry_records` make the catalog
queryable from IDEs and from agents at runtime.

In [ ]:
listed = aws_call(
    registry_data,
    "list_discoverable_registry_records",
    registryId=registry_id,
    maxResults=20,
    filters=[{"name": "recordType", "values": ["AGENT"]}],
)
print(f"discoverable AGENT records: {len(listed.get('registryRecords', []))}")
for item in listed.get("registryRecords", []):
    print(f"  {item['recordId']}  {item['name']}  v{item['recordVersion']}  {item['status']}")

found = aws_call(
    registry_data,
    "search_discoverable_registry_records",
    searchQuery="demonstration agent for a security lab",
    registryIds=[registry_id],
    maxResults=5,
)
print(f"search hits: {[hit['name'] for hit in found.get('registryRecords', [])]}")

batch = aws_call(
    registry_data,
    "batch_get_discoverable_registry_record",
    entries=[{"registryId": registry_id, "recordIds": [record_id]}],
)
for item in batch.get("registryRecords", []):
    card_text = ((item.get("descriptors") or {}).get("a2aAgentCard") or {}).get("data", "{}")
    print(f"agent card for {item['name']}:")
    pretty(json.loads(card_text), limit=1500)
if not batch.get("registryRecords") and not DRY_RUN:
    print("nothing returned: only APPROVED records are discoverable; check the status printed in A.4")

# Part B: Gemini Enterprise Agent Platform

Google announced Gemini Enterprise Agent Platform at Cloud Next in April 2026 as the successor
to Vertex AI Agent Builder: Agent Development Kit (ADK), **Agent Runtime** (formerly Agent Engine,
still the `reasoningEngines` API), **Agent Registry**, Agent Identity, Agent Gateway and Agent
Observability. The registry is the Agent Registry API (`agentregistry.googleapis.com`, v1, Python
package `google-cloud-agentregistry`):

* You **write** a `Service` under `projects/*/locations/*/services`. Its spec decides what it
  describes: `agent_spec` (type `A2A_AGENT_CARD` with the card inline, or `NO_SPEC` plus
  `interfaces` for an agent that is not A2A), `mcp_server_spec` or `endpoint_spec`.
* The registry **derives** read-only `Agent`, `McpServer` and `Endpoint` resources
  (`service.registry_resource` names the one you created). Consumers only ever read those.
* Agents deployed to Agent Runtime are **registered automatically**; their `Agent` carries
  `agentregistry.googleapis.com/system/RuntimeReference` and `.../Framework` attributes.
* `agents.search` understands field queries such as `skills.tags:lab` or `displayName:nexus*`, and the
  registry doubles as an MCP server for ADK and other agents.
* Access is IAM: `roles/agentregistry.viewer` to read, `roles/agentregistry.admin` to register.
  Bindings link an agent to an Agent Identity auth provider; Agent Gateway enforces policies at
  runtime.

The Gemini Enterprise **app** (formerly Agentspace) is a separate surface: an agent becomes visible
to end users when it is registered on the app's assistant through the Discovery Engine API (B.4),
either from Agent Registry or directly.

In [ ]:
import google.auth
from google.api_core.exceptions import GoogleAPICallError, NotFound
from google.cloud import agentregistry_v1 as ar

if DRY_RUN:
    gcp_credentials = None
    GCP_PROJECT = GCP_PROJECT or "example-project"
    gar_client = None
    print("dry run: no Google credentials loaded")
else:
    gcp_credentials, adc_project = google.auth.default(
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )
    GCP_PROJECT = GCP_PROJECT or adc_project or ""
    if not GCP_PROJECT:
        raise RuntimeError("set GOOGLE_CLOUD_PROJECT: Application Default Credentials carry no project")
    gar_client = ar.AgentRegistryClient(credentials=gcp_credentials)

GCP_PARENT = f"projects/{GCP_PROJECT}/locations/{GCP_LOCATION}"
print(f"Agent Registry parent: {GCP_PARENT}")

## B.1 Create the agent on Agent Runtime (optional)

Set `GCP_DEPLOY_AGENT=1` and `GCP_STAGING_BUCKET=gs://...` to deploy a minimal ADK agent. The
`agentplatform` client (shipped with `google-cloud-aiplatform`; older releases expose the same
client as `vertexai.Client`) packages the agent, builds it and creates a `reasoningEngines`
resource. Deployment takes several minutes. Set `GCP_ENGINE_NAME` instead to reuse a deployment.

Deployed agents are registered automatically; the next cell waits until the registry lists an
`Agent` whose runtime reference points at the new reasoning engine.

In [ ]:
gcp_engine_name = os.environ.get("GCP_ENGINE_NAME", "")
platform = None

if GCP_DEPLOY_AGENT and not DRY_RUN:
    if not GCP_STAGING_BUCKET.startswith("gs://"):
        raise RuntimeError("GCP_STAGING_BUCKET must be a gs:// bucket in the same project")
    from google.adk.agents import Agent as AdkAgent

    try:
        import agentplatform as platform_sdk
        from agentplatform.agent_engines import AdkApp
    except ImportError:  # older SDK releases expose the same client under vertexai
        import vertexai as platform_sdk
        from vertexai.agent_engines import AdkApp

    root_agent = AdkAgent(
        name=AGENT_NAME.replace("-", "_"),
        model=GCP_MODEL,
        description="Demonstration agent for the Project Nexus registry lab. It answers short questions.",
        instruction="You are a demonstration agent in a security lab. Answer in one short sentence.",
    )
    platform = platform_sdk.Client(project=GCP_PROJECT, location=GCP_LOCATION)
    remote = platform.agent_engines.create(
        agent=AdkApp(agent=root_agent, enable_tracing=False),
        config={
            "display_name": AGENT_NAME,
            "description": "Demonstration ADK agent created by the Project Nexus registry lab.",
            "staging_bucket": GCP_STAGING_BUCKET,
            "requirements": ["google-cloud-aiplatform[agent_engines,adk]"],
            "agent_framework": "google-adk",
        },
    )
    gcp_engine_name = remote.api_resource.name
    print(f"Agent Runtime resource: {gcp_engine_name}")
elif GCP_DEPLOY_AGENT:
    print("dry run: the Agent Runtime deployment is skipped")
elif gcp_engine_name:
    print(f"reusing {gcp_engine_name}")
else:
    print("GCP_DEPLOY_AGENT is off: Part B registers an A2A agent card without a live endpoint")

In [ ]:
RUNTIME_REFERENCE = "agentregistry.googleapis.com/system/RuntimeReference"


def runtime_reference(agent: dict[str, Any]) -> str:
    """The URI of the resource hosting an auto-registered agent, if any."""
    attributes = agent.get("attributes") or {}
    return str((attributes.get(RUNTIME_REFERENCE) or {}).get("uri", ""))


auto_registered: dict[str, Any] | None = None
if gcp_engine_name and gar_client is not None:
    engine_id = gcp_engine_name.rsplit("/", 1)[-1]

    def find_registered() -> dict[str, Any]:
        for agent in gar_client.list_agents(parent=GCP_PARENT):
            info = ar.Agent.to_dict(agent)
            if engine_id in runtime_reference(info):
                return {"status": "REGISTERED", "agent": info}
        return {"status": "PENDING"}

    auto_registered = wait_for(
        find_registered,
        lambda state: state["status"] == "REGISTERED",
        what="automatic registration",
        timeout=600,
        interval=20,
    )["agent"]
    pretty(auto_registered, limit=2500)
else:
    print("no Agent Runtime deployment to look up")

## B.2 Register an A2A agent manually (Service → Agent)

Manual registration is how you catalog agents that run elsewhere: on Cloud Run, GKE, another cloud
or on premises. Create a `Service` whose `agent_spec` is the A2A agent card; the registry validates
the card (10 KB limit) and derives the `Agent`. For the `A2A_AGENT_CARD` type the connection details
come from the card's `url`, so `interfaces` must stay empty. The create call is a long-running
operation.

In [ ]:
gcp_agent_url = os.environ.get("GCP_AGENT_URL") or f"https://{PREFIX}.example.invalid/agents/{AGENT_NAME}-gcp"
gcp_agent_card = build_agent_card(
    name=f"{AGENT_NAME}-gcp",
    description="Demonstration A2A agent created by the Project Nexus registry lab (Google Cloud).",
    url=gcp_agent_url,
    version="1.0.0",
    skill_tags=["nexus-lab", "a2a", "demo"],
)
service_id = f"{PREFIX}-a2a-{RUN_ID}"[:63]  # 4-63 characters: lowercase letters, digits and hyphens
service = ar.Service(
    display_name=f"{PREFIX} A2A agent {RUN_ID}"[:63],
    description="Demonstration A2A agent registered by the Project Nexus agent-registry lab.",
    agent_spec=ar.Service.AgentSpec(type_=ar.Service.AgentSpec.Type.A2A_AGENT_CARD, content=gcp_agent_card),
)
print(f"POST {GCP_PARENT}/services?serviceId={service_id}")
print(ar.Service.to_json(service, indent=2)[:1500])

gcp_service_name = f"{GCP_PARENT}/services/{service_id}"
gcp_registry_agent: dict[str, Any] | None = None
if gar_client is not None:
    operation = gar_client.create_service(parent=GCP_PARENT, service_id=service_id, service=service)
    created_service = operation.result(timeout=300)
    gcp_service_name = created_service.name
    print(f"service {gcp_service_name}\nderived resource {created_service.registry_resource}")
    gcp_registry_agent = ar.Agent.to_dict(gar_client.get_agent(name=created_service.registry_resource))
    pretty(gcp_registry_agent, limit=2500)

## B.3 Discover agents

`agents.list` returns every agent in the location; `agents.search` accepts field queries. Both only
return the read-only `Agent` resources, never the writable services, which is what keeps consumer
permissions (`roles/agentregistry.viewer`) separate from publisher permissions.

In [ ]:
if gar_client is None:
    print("dry run: list and search are skipped")
else:
    print(f"agents in {GCP_PARENT}:")
    for agent in gar_client.list_agents(parent=GCP_PARENT):
        protocols = [protocol.type_.name for protocol in agent.protocols]
        print(f"  {agent.name}  {agent.display_name!r}  protocols={protocols}")
    print("\nsearch skills.tags:nexus-lab")
    for agent in gar_client.search_agents(parent=GCP_PARENT, search_string="skills.tags:nexus-lab"):
        print(f"  {agent.name}  {agent.display_name!r}  version={agent.version!r}")

## B.4 Publish the agent in a Gemini Enterprise app (optional)

End users meet agents inside a Gemini Enterprise app. Registering there is a Discovery Engine API
call on the app's default assistant:

```
POST https://discoveryengine.googleapis.com/v1alpha/projects/PROJECT_NUMBER/locations/global
     /collections/default_collection/engines/APP_ID/assistants/default_assistant/agents
```

The body is an `Agent` with `displayName`, `description` and one definition: `a2aAgentDefinition`
(`jsonAgentCard`, the card as a string) for an A2A endpoint, or `adkAgentDefinition`
(`provisionedReasoningEngine.reasoningEngine`) for an Agent Runtime deployment. Apps in the `us` or
`eu` multi-region use `us-` or `eu-discoveryengine.googleapis.com`. The console also offers *Import
from Agent Registry*, which pulls an A2A agent already cataloged in B.2. Set `GE_APP_ID` to run this
cell; there is no public client library for this alpha surface, so it uses the REST API directly.

In [ ]:
from google.auth.transport.requests import AuthorizedSession


def ge_host(location: str) -> str:
    """Discovery Engine host for a Gemini Enterprise app location."""
    if location == "global":
        return "discoveryengine.googleapis.com"
    return f"{location}-discoveryengine.googleapis.com"


ge_session = None
ge_agent_name = ""
if GE_APP_ID and not DRY_RUN:
    ge_project = GE_PROJECT_NUMBER or GCP_PROJECT
    ge_parent = (
        f"projects/{ge_project}/locations/{GE_LOCATION}/collections/default_collection"
        f"/engines/{GE_APP_ID}/assistants/default_assistant"
    )
    ge_body: dict[str, Any] = {
        "displayName": f"Nexus lab agent {RUN_ID}",
        "description": "Demonstration agent registered by the Project Nexus registry lab.",
    }
    if gcp_engine_name:
        ge_body["adkAgentDefinition"] = {"provisionedReasoningEngine": {"reasoningEngine": gcp_engine_name}}
    else:
        ge_body["a2aAgentDefinition"] = {"jsonAgentCard": json.dumps(gcp_agent_card)}
    ge_session = AuthorizedSession(gcp_credentials)
    ge_response = ge_session.post(
        f"https://{ge_host(GE_LOCATION)}/v1alpha/{ge_parent}/agents",
        json=ge_body,
        headers={"X-Goog-User-Project": GCP_PROJECT},
        timeout=60,
    )
    if ge_response.status_code >= 400:
        print(ge_response.status_code, ge_response.text[:1500])
    ge_response.raise_for_status()
    ge_agent = ge_response.json()
    ge_agent_name = ge_agent.get("name", "")
    pretty(ge_agent)
elif GE_APP_ID:
    print("dry run: the Gemini Enterprise registration is skipped")
else:
    print("GE_APP_ID is empty: skipping the Gemini Enterprise app registration")

# Part C: What ShadowScan sees, and reconciling against an inventory

A registry entry and a running agent are different facts. ShadowScan's cloud connectors observe
**deployed resources**: `cloud.aws` lists AgentCore runtimes, gateways, memories and workload
identities (service `agentcore`), `cloud.gcp` lists Agent Runtime deployments (`reasoningEngines`),
Discovery Engine apps and more. At the time of writing neither connector reads the two registries
themselves, so:

* the runtimes you deployed in A.1 and B.1 appear as `agent` findings, and are **shadow** until an
  inventory entry claims them;
* the registry records from A.3 and B.2 are not findings. A registered agent that is not deployed is
  invisible to the scanner, and a deployed agent that nobody registered is exactly what the scanner
  is for.

The cells below write one Agent Capability Card per lab agent into `lab-output/inventory/`. The
`discovery.resources` patterns bind the runtime identifiers ShadowScan reports (`agentRuntimeArn`
for AgentCore, the `projects/.../reasoningEngines/...` name for Agent Runtime). The registry
identifiers are recorded in `capability_surface` for the reviewer; they do not match any finding today.

`RUN_SHADOWSCAN=1` then runs the two connectors against the same account and project. Use a
**separate, read-only identity** for that scan (for example a dedicated profile exported as
`AWS_PROFILE` and a read-only service account for ADC): ShadowScan's trust model keeps write
credentials away from scanning, and the lab's deploy credentials are far broader than a scan needs.

In [ ]:
import yaml

INVENTORY_DIR = OUTPUT_DIR / "inventory"
INVENTORY_DIR.mkdir(parents=True, exist_ok=True)


def write_card(
    *, agent_id: str, provider: str, resources: list[str], names: list[str], registry_refs: dict[str, str]
) -> Path:
    """Write an Agent Capability Card that approves the given discovered resources."""
    card = {
        "metadata": {
            "agent_id": agent_id,
            "version": "1.0.0",
            "owner_team": "security-lab",
            "classification": "Internal",
        },
        "autonomy_profile": {"level": 1, "memory_persistence": False, "max_loop_iterations": 5},
        "capability_surface": {
            "authorized_tools": [],
            "registry_references": registry_refs,  # informational: not matched by ShadowScan
        },
        "security_controls": {"egress_proxy_required": False, "kill_switch_enabled": True},
        "discovery": {
            "resources": resources,
            "names": names,
            "surfaces": ["cloud"],
            "providers": [provider],
        },
    }
    path = INVENTORY_DIR / f"{agent_id}.yaml"
    path.write_text(yaml.safe_dump(card, sort_keys=False), encoding="utf-8")
    return path


aws_card_path = write_card(
    agent_id=f"{AGENT_NAME}-aws",
    provider="aws",
    resources=[aws_runtime_arn] if aws_runtime_arn else [],
    names=[AGENT_NAME, "Nexus lab agent"],
    registry_refs={"registry_arn": registry_arn, "record_arn": record_arn},
)
gcp_card_path = write_card(
    agent_id=f"{AGENT_NAME}-gcp",
    provider="gcp",
    resources=[gcp_engine_name] if gcp_engine_name else [],
    names=[f"{AGENT_NAME}-gcp", AGENT_NAME.replace("-", "_")],
    registry_refs={"service": gcp_service_name, "gemini_enterprise_agent": ge_agent_name},
)
for path in (aws_card_path, gcp_card_path):
    print(f"--- {path}")
    print(path.read_text(encoding="utf-8"))
print("cards without resources can only produce review suggestions; deploy an agent to bind one")

In [ ]:
import shutil
import subprocess


def run_shadowscan(arguments: list[str], report: Path) -> None:
    """Run one ShadowScan connector against the lab inventory and keep the JSON report."""
    command = ["shadowscan", *arguments, "--inventory", str(INVENTORY_DIR), "--format", "json"]
    command += ["-o", str(report)]
    print("$ " + " ".join(command))
    completed = subprocess.run(command, check=False, capture_output=True, text=True, timeout=1800)
    print(completed.stdout[-3000:])
    print(completed.stderr[-2000:])
    print(f"exit {completed.returncode} (0 complete, 2 findings at or above --fail-on, 3 incomplete scan)")
    if report.exists():
        findings = json.loads(report.read_text(encoding="utf-8")).get("findings", [])
        for finding in findings:
            line = (
                f"  {finding.get('kind'):14s} shadow={finding.get('shadow')!s:5s} {finding.get('resource')}"
            )
            print(line)


if not RUN_SHADOWSCAN:
    print("RUN_SHADOWSCAN is off; set it to 1 (and use a read-only identity) to scan the lab account/project")
elif DRY_RUN:
    print("dry run: ShadowScan is not executed")
elif shutil.which("shadowscan") is None:
    print("shadowscan is not on PATH; install it from a reviewed checkout: pip install -e '.[aws,gcp]'")
else:
    run_shadowscan(
        ["run", "cloud.aws", "--set", f"regions={AWS_REGION}", "--set", "services=agentcore"],
        OUTPUT_DIR / "shadowscan-aws.json",
    )
    run_shadowscan(
        ["run", "cloud.gcp", "--set", f"projects={GCP_PROJECT}", "--set", f"locations={GCP_LOCATION}"],
        OUTPUT_DIR / "shadowscan-gcp.json",
    )

# Part D: Cleanup

Set `LAB_CLEANUP=1` and run the two cells below. Order matters: records before the registry on AWS,
the Gemini Enterprise registration before the Agent Registry service on Google Cloud, and runtimes
last. The AgentCore toolkit's `destroy()` removes the runtime, its endpoint and the resources it
created for the deployment; review `resources_removed` and `errors` in its result.

In [ ]:
def aws_error_code(exc: ClientError) -> str:
    return str(exc.response.get("Error", {}).get("Code", ""))


def record_state() -> dict[str, Any]:
    try:
        return aws_call(registry_control, "get_registry_record", registryId=registry_id, recordId=record_arn)
    except ClientError as exc:
        if aws_error_code(exc) == "ResourceNotFoundException":
            return {"status": "DELETED"}
        raise


def registry_state() -> dict[str, Any]:
    try:
        return aws_call(registry_control, "get_registry", registryId=registry_id)
    except ClientError as exc:
        if aws_error_code(exc) == "ResourceNotFoundException":
            return {"status": "DELETED"}
        raise


if not LAB_CLEANUP:
    print("LAB_CLEANUP is off; nothing deleted")
else:
    try:
        aws_call(registry_control, "delete_registry_record", registryId=registry_id, recordId=record_arn)
    except ClientError as exc:
        if aws_error_code(exc) != "ResourceNotFoundException":
            raise
    if not DRY_RUN:
        wait_for(
            record_state, lambda state: state["status"] == "DELETED", what="record deletion", timeout=300
        )
    try:
        aws_call(registry_control, "delete_registry", registryId=registry_id)
    except ClientError as exc:
        if aws_error_code(exc) != "ResourceNotFoundException":
            raise
    if not DRY_RUN:
        wait_for(
            registry_state, lambda state: state["status"] == "DELETED", what="registry deletion", timeout=300
        )
    if aws_runtime is not None:
        destroyed = aws_runtime.destroy()
        print(f"runtime resources removed: {destroyed.resources_removed}")
        if destroyed.errors:
            print(f"destroy errors: {destroyed.errors}")
    print("AWS cleanup finished")

In [ ]:
if not LAB_CLEANUP:
    print("LAB_CLEANUP is off; nothing deleted")
elif DRY_RUN:
    print("dry run: Google Cloud cleanup is skipped")
else:
    if ge_agent_name and ge_session is not None:
        deleted = ge_session.delete(
            f"https://{ge_host(GE_LOCATION)}/v1alpha/{ge_agent_name}",
            headers={"X-Goog-User-Project": GCP_PROJECT},
            timeout=60,
        )
        print(f"Gemini Enterprise agent delete: HTTP {deleted.status_code}")
    try:
        gar_client.delete_service(name=gcp_service_name).result(timeout=300)
        print(f"deleted {gcp_service_name}")
    except NotFound:
        print(f"{gcp_service_name} was already gone")
    except GoogleAPICallError as exc:
        print(f"service deletion failed: {exc.message}")
        raise
    if platform is not None and gcp_engine_name:
        platform.agent_engines.delete(name=gcp_engine_name, force=True)
        print(f"deleted Agent Runtime deployment {gcp_engine_name}")
    print("Google Cloud cleanup finished")

# Wrap-up

What you exercised, side by side:

| Step | AWS Agent Registry | Gemini Enterprise Agent Platform |
|---|---|---|
| Create the agent | AgentCore Runtime, A2A protocol, direct code deploy | Agent Runtime (`reasoningEngines`), ADK agent |
| Describe it | A2A agent card as `AGENT` record descriptor | A2A agent card as `Service.agent_spec` |
| Register | `CreateRegistry` + `CreateRegistryRecord` | `services.create` (long-running operation) |
| Govern | submit → curator approval, `DEPRECATED`, EventBridge hooks, custom metadata schema | IAM roles, automatic registration, Agent Gateway and policies, bindings to Agent Identity |
| Discover | data-plane list, search and batch get; registry MCP endpoint | `agents.list`, `agents.search`; Agent Registry MCP server; Gemini Enterprise app |

Takeaways for a discovery program:

* A registry is an **approval list**, not a sensor. Both platforms will happily hold records for
  agents that no longer exist, and neither sees agents that were never submitted.
* Automatic population (AWS auto-detection, Google automatic registration) narrows the gap for the
  platform's own runtimes only. Agents on Lambda, Cloud Run, Kubernetes or a laptop still need an
  independent scan.
* Reconciliation needs stable identifiers. Bind inventory entries to runtime ARNs and resource names,
  not to display names, and record the registry identifiers next to them for the reviewer.

Further reading (official documentation):

* AWS: [AWS Agent Registry](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/registry.html),
  [record lifecycle](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/registry-record-lifecycle.html),
  [IAM permissions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/registry-iam-permissions.html),
  [registry MCP endpoint](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/registry-mcp-endpoint.html),
  [A2A servers on AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-a2a.html),
  [boto3 `agent-registry-control`](https://docs.aws.amazon.com/boto3/latest/reference/services/agent-registry-control.html).
* Google Cloud: [Agent Registry overview](https://docs.cloud.google.com/agent-registry/overview),
  [data model](https://docs.cloud.google.com/agent-registry/data-model),
  [register agents](https://docs.cloud.google.com/agent-registry/register-agents),
  [search](https://docs.cloud.google.com/agent-registry/search-agents-and-tools),
  [Gemini Enterprise Agent Platform agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/agents),
  [register A2A agents in Gemini Enterprise](https://docs.cloud.google.com/gemini/enterprise/docs/register-and-manage-an-a2a-agent),
  [register ADK agents hosted on Agent Runtime](https://docs.cloud.google.com/gemini/enterprise/docs/register-and-manage-an-adk-agent),
  [`google-cloud-agentregistry` on PyPI](https://pypi.org/project/google-cloud-agentregistry/).
* ShadowScan: [cloud connectors](https://github.com/aisecnomad/Project-Nexus/blob/main/docs/connectors/cloud.md),
  [sanctioned inventory](https://github.com/aisecnomad/Project-Nexus/blob/main/docs/inventory.md).